In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import shii
import geopandas as gpd
import pandas as pd
import os
from shapely import Point
import datetime as dt
import matplotlib.pyplot as plt

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, root_mean_squared_error
from sklearn.inspection import permutation_importance



# 1. Prep data

In [ ]:
weather_df = shii.download_weather('2010-01-01T00:00:00', '2025-12-31T23:59:59')
hvi_df = shii.download_hvi()
cdta = shii.download_community_districts()
cdta_population = shii.download_cd_population()

In [ ]:
# weather_df.to_csv('./data/weather.csv')
# hvi_df.to_csv('./data/hvi.csv')
# cdta_population.to_csv('./data/cdta_pop.csv')
# cdta.set_crs('EPSG:4326').to_file('./data/cdta.gpkg')

In [ ]:
cdta_borough_lookup = {
    'MN':1,
    'Manhattan': 1,
    'BX': 2,
    'Bronx': 2,
    'BK': 3,
    'Brooklyn': 3,
    'QN': 4,
    'Queens': 4,
    'SI': 5,
    'Staten Island': 5
}

In [ ]:
hvi_df['borough'] = hvi_df['CDTACode'].astype(str).str[:2]
hvi_df['borough_code'] = hvi_df['borough'].map(cdta_borough_lookup)
hvi_df['cdta'] = hvi_df['borough_code'].astype(str) + hvi_df['CDTACode'].astype(str).str[-2:]
cdta_population['borough_code'] = cdta_population['borough'].map(cdta_borough_lookup)
cdta_population['cdta'] = cdta_population['borough_code'].astype(str) + cdta_population['cd_number'].str.zfill(2)

In [ ]:
all_311_df = gpd.read_file('./311_calls.gpkg')
cdta = cdta.set_crs('EPSG:4326')
all_311_df = all_311_df.sjoin(cdta[['geometry', 'boro_cd']]) 
# Separate FHE from rest of hydrant data
all_311_df.loc[(all_311_df['descriptor']=='Fire Hydrant Emergency (FHE)'), 'complaint_type'] = 'fhe'

In [ ]:
all_311_df.groupby('complaint_type').size()

In [ ]:
heat_incidents.groupby('final_call_type').size()

In [ ]:
(heat_incidents['final_call_type']!='HEAT').sum()

In [ ]:
heat_incidents = pd.read_csv('./ems_calls.csv')
heat_incidents = heat_incidents.loc[~heat_incidents['communitydistrict'].isna()]
heat_incidents['communitydistrict'] = heat_incidents['communitydistrict'].astype(int).astype(str)
# heat_incidents = heat_incidents.loc[heat_incidents['final_severity_level_code'] >= 4]
heat_incidents = heat_incidents.loc[heat_incidents['final_call_type'] == 'HEAT']

# 2. Spatiotemporal merge

In [ ]:
rolling_days = 2
heat_incidents['datetime'] = pd.to_datetime(heat_incidents['incident_datetime'])
heat_incidents['date'] = pd.to_datetime(heat_incidents['datetime'].dt.date)
heat_inc_rolling = (heat_incidents
                    .groupby(['date', 'communitydistrict'])
                    .size()
                    .rename('heat_ems_count')
                    .reset_index()
                    .set_index('date')
                    .groupby('communitydistrict')
                    .rolling(window=dt.timedelta(rolling_days), min_periods=0)
                    .sum())
heat_inc_rolling = heat_inc_rolling.reset_index()
# heat_inc_rolling['window_end'] = heat_inc_rolling['date']
heat_inc_rolling['date'] = heat_inc_rolling['date'] - dt.timedelta(rolling_days)
heat_inc_rolling = heat_inc_rolling.rename(columns={'communitydistrict':'cdta'})
heat_inc_rolling = heat_inc_rolling.set_index(['cdta','date'])

In [ ]:
cdta_311_counts = all_311_df.groupby(['boro_cd', 'date', 'complaint_type']).size()
cdta_311_counts.name = '311_counts'
cdta_311_counts = cdta_311_counts.reset_index()
cdta_311_counts = cdta_311_counts.rename(columns={'boro_cd':'cdta'})
cdta_311_counts = cdta_311_counts.pivot(index=['cdta','date'], columns='complaint_type')['311_counts'].fillna(0).reset_index()
# cdta_311_counts = cdta_311_counts.merge(cdta_population[['cdta', '_2010_population']], on='cdta').rename(columns={'_2010_population': 'population'})
# cdta_311_counts['population'] = cdta_311_counts['population'].astype(float)

In [ ]:
rolling_days=3
# for type in ['ac','hydrant','power','ventilation']
cdta_311_rolling = (cdta_311_counts
                    .set_index('date')
                    .groupby('cdta')
                    .rolling(window=dt.timedelta(rolling_days), min_periods=0)
                    .sum())
cdta_311_rolling = cdta_311_rolling.reset_index()

In [ ]:
all_dates = weather_df.index.unique()
all_cdtas = cdta['boro_cd'].unique()

# 3. Create a MultiIndex of every combination (Date x Tract)
full_index = pd.MultiIndex.from_product([all_dates, all_cdtas], names=['date', 'cdta'])
cdta_311_complete = cdta_311_rolling.set_index(['date', 'cdta']).reindex(full_index, fill_value=0).reset_index()
# cdta_311_complete = cdta_311_counts.set_index(['date', 'cdta']).reindex(full_index, fill_value=0).reset_index()

In [ ]:
cdta_311_weather = cdta_311_complete.merge(weather_df, left_on='date', right_on='time',how='left')

In [ ]:
hvi_df = hvi_df.groupby('cdta')[['HVI_RANK','SURFACE_TEMP','MEDIAN_INCOME','GREENSPACE','PCT_HOUSEHOLDS_AC','PCT_BLACK_POP']].mean()

In [ ]:
hvi_df = hvi_df.reset_index().merge(cdta_population[['cdta', '_2010_population']], on='cdta').rename(columns={'_2010_population': 'population'})
hvi_df['population'] = hvi_df['population'].astype(float)

In [ ]:
cdta_311_hvi_weather = cdta_311_weather.merge(hvi_df, on='cdta', how='inner')

In [ ]:
cdta_alldata = cdta_311_hvi_weather.set_index(['cdta', 'date']).join(heat_inc_rolling)
cdta_alldata['population'] = cdta_alldata['population'].astype(float)
# cdta_alldata['heat_ems_count']= cdta_alldata['heat_ems_count'].fillna(0)
cdta_alldata = cdta_alldata.fillna(0)

In [ ]:
for col in ['ac','hydrant','power','ventilation','fhe', 'heat_ems_count']:
    cdta_alldata[col+'_norm'] = 100000*cdta_alldata[col]/cdta_alldata['population']

# 3. RF regression

In [ ]:
def filter_date(df, year_start, year_end):
    df_years = df.index.get_level_values(1).year  
    filt_index = (df_years >= year_start) & (df_years < year_end)
    return df.loc[filt_index]

In [ ]:
# Define predictor sets
pred_sets = {
    '311': ['fhe_norm', 'hydrant_norm', 'ventilation_norm','power_norm', 'ac_norm'],
    'hvi': ['HVI_RANK','GREENSPACE','MEDIAN_INCOME','SURFACE_TEMP','PCT_HOUSEHOLDS_AC', 'PCT_BLACK_POP'],
    'weather': ['tmin','tmax','rhum','wspd','cldc','pres'],
    'best': ['fhe_norm','hydrant_norm','ventilation_norm','tmin','tmax','rhum', 'pres','wspd','GREENSPACE','MEDIAN_INCOME'],
    'best_no311': ['tmin','tmax','rhum', 'pres','wspd','GREENSPACE','MEDIAN_INCOME']
}

In [ ]:
# Select predictors:
# Everything, R2 around 0.106, 0.148 on shorter
# X = cdta_alldata[pred_sets['311'] + pred_sets['hvi'] + pred_sets['weather']]
# 311 only - ok, R2 0.0485
# X = cdta_alldata[pred_sets['311']]
# Weather only, bad R2 0.027
# X = cdta_alldata[pred_sets['weather']]
# HVI only, R2 0.029
# X = cdta_alldata[pred_sets['hvi']]
# Weather + HVI: R2 0.109, 0.142 on shorter horizon
# X = cdta_alldata[pred_sets['weather'] + pred_sets['hvi']]
# Weather + 311, R2 0.042, weird that it's worse than 311 by itself
# X = cdta_alldata[pred_sets['weather'] + pred_sets['311']]
# HVI + 311, R2 0.07
# X = cdta_alldata[pred_sets['hvi'] + pred_sets['311']]
# 'All-stars', R2 0.107 (0.15 on shorter time horizon, 3 days 311 and 2 days heat), 0.17 with bigger model
X = cdta_alldata[pred_sets['best']]
# all-stars without 311, R2 of 0.155
# X = cdta_alldata[pred_sets['best_no311']]

y = cdta_alldata['heat_ems_count_norm']

# Get valid data
valid_locs = (X.isna().sum(axis=1)==0)*(~y.isna())
X = X.loc[valid_locs]
y = y.loc[valid_locs]

# Split the data into training and testing sets
val_start = 2023
test_start = 2025
total_examples = np.sum(valid_locs)
X_train, y_train = filter_date(X, 0, val_start), filter_date(y, 0, val_start)
X_val, y_val = filter_date(X, val_start, test_start), filter_date(y, val_start, test_start)
X_test, y_test = filter_date(X, test_start, 9999), filter_date(y, test_start, 9999)
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, train_size=round(total_examples*0.8), test_size=round(total_examples*0.2), random_state=7
# )


In [ ]:
# Initialize the regressor
# n_estimators is the number of decision trees in the forest
regr = RandomForestRegressor(n_estimators=200, random_state=0, max_depth=10)

# Train the model on the training data
regr.fit(X_train, y_train)


In [ ]:
# Predict on the test data
y_pred = regr.predict(X_val)

In [ ]:
# Calculate evaluation metrics
mse = mean_squared_error(y_val, y_pred)
rmse = root_mean_squared_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)

print(f"Root Mean Squared Error: {rmse}")
print(f"Mean Squared Error: {mse}")
print(f"R-squared Score: {r2}")


In [ ]:
plt.scatter(y_pred, y_val)

In [ ]:
importances = regr.feature_importances_
std = np.std([tree.feature_importances_ for tree in regr.estimators_], axis=0)

In [ ]:
feature_names = X.columns

In [ ]:
import pandas as pd

forest_importances = pd.Series(importances, index=feature_names)

fig, ax = plt.subplots()
forest_importances.plot.bar(yerr=std, ax=ax)
ax.set_title("Feature importances using MDI")
ax.set_ylabel("Mean decrease in impurity")
fig.tight_layout()

# Building map of predictor over time in 2023-2024

In [ ]:
y_pred_df = pd.DataFrame(y_val.rename('true'))
y_pred_df['pred'] = y_pred
cdta_pred_df = cdta.rename(columns={'boro_cd':'cdta'}).set_index('cdta').join(y_pred_df).reset_index().set_index('date')

In [ ]:
# Source - https://stackoverflow.com/a/78609847
# Posted by Trenton McKinney, modified by community. See post 'Timeline' for change history
# Retrieved 2026-03-10, License - CC BY-SA 4.0

from matplotlib.animation import FuncAnimation, PillowWriter

# Initialize the plot
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(10, 5))

# Set fixed axis limits
xlim = (cdta_pred_df.total_bounds[0], cdta_pred_df.total_bounds[2])
ylim = (cdta_pred_df.total_bounds[1], cdta_pred_df.total_bounds[3])

for ax in axs:
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    # Plot initial boundaries
    boundary = cdta.boundary.plot(ax=ax, edgecolor='black', linewidth=0.3)

    # Initialize the colorbar variable with a fixed normalization
    norm = plt.Normalize(vmin=0, vmax=3)
    sm = plt.cm.ScalarMappable(cmap='RdBu_r', norm=norm)
    sm.set_array([])  # Only needed for adding the colorbar
    colorbar = fig.colorbar(sm, ax=ax, orientation='horizontal', shrink=0.5, format='%.2f')

# Function to update the plot for each year
def animate(date):
    for ax in axs:
        ax.clear()
        # Set the fixed axis limits
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        # Plot initial boundaries
        boundary = cdta.boundary.plot(ax=ax, edgecolor='black', linewidth=0.3)
    
    # Plot the data for the current year
    cdta_pred_df.loc[date].plot(
        ax=axs[0], column='true', legend=False, cmap='RdBu_r', norm=norm
    )
    cdta_pred_df.loc[date].plot(
        ax=axs[1], column='pred', legend=False, cmap='RdBu_r', norm=norm
    )

    # Add year annotation at the top
    axs[0].annotate(f'True, Date: {date}', xy=(0.5, 1.05), xycoords='axes fraction', fontsize=12, ha='center')
    axs[1].annotate(f'Pred, Date: {date}', xy=(0.5, 1.05), xycoords='axes fraction', fontsize=12, ha='center')

# Create the animation
dates = pd.date_range(cdta_pred_df.index.values[0], cdta_pred_df.index.values[-1])
animation = FuncAnimation(fig, animate, frames=dates, repeat=False, interval=20)

# Save the animation as a GIF
writer = PillowWriter(fps=5)
animation.save('test.gif', writer=writer)

# Show the plot
plt.show()
